# SDG Dry-Run: 100-Example Format Validation (D-19)

This notebook runs a ~100-example generation across all 5 curriculum tiers to validate
the OSFT output format before committing to the full production run (8,500-14,000 examples).

**Purpose:** Produce ~100 tool-calling traces spanning single_tool, parallel, sequential,
multi_turn, and closed_loop tiers, serialize them to both single-turn and multi-turn OSFT
JSONL, and validate the schema so Farbod's team can confirm the format works with Training
Hub's `osft()` call.

**Why all 5 tiers, not just single_tool:** there are only 7 single_tool seed scenarios
(21 total across all tiers) -- reaching ~100 examples requires sampling multiple generated
variations per seed (via sdg_hub's `multiply_tool_rows`/`sample_tools` runtime params) AND
drawing from every tier. This also means the dry-run exercises the multi-turn serialization
path (multi_turn/closed_loop tiers), not just single-turn -- validating D-20 (both formats)
before the full run, which the original 10-single_tool-only design did not do.

**Prerequisites:**
- vLLM serving Gemma 4 31B is accessible (set `VLLM_API_KEY` and `VLLM_API_BASE`)
- Langflow agent is deployed and accessible (set `LANGFLOW_API_KEY` and `LANGFLOW_URL`)
- MCP Gateway is reachable from this cluster (set `GATEWAY_URL`, `KEYCLOAK_URL`, `CLIENT_SECRET`)
- This notebook is intended to run in an RHOAI workbench pointed at the `OTel_agentic_improvements`
  repo (e.g. the `sdg-pipeline` workbench in namespace `otel25-sdg`)

**D-19 Gate:** The dry-run output must be sent to Farbod's team. They run
`osft(data_path="dry-run.jsonl")` and confirm it tokenizes correctly and loss values are normal.
Do NOT proceed to the full production run until they confirm.

In [ ]:
import sys
import os
import json
import logging

# Add repo root to path for proper package imports
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
# If running from sdg/notebooks/, this resolves to the repo root
sys.path.insert(0, os.path.abspath('../..'))

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# ── Config Loading with env-var interpolation ──
# yaml.safe_load() does NOT expand ${VAR} placeholders.
# os.path.expandvars() ALSO does not support bash's ${VAR:-default} syntax --
# it only matches plain ${VAR}/$VAR, so every agent_config/model_config value
# using the `:-default` form was silently passed through as the literal
# unexpanded string regardless of whether the env var was set. Found live:
# a real generation run hit `httpx.UnsupportedProtocol: Request URL is
# missing an 'http://' or 'https://' protocol` because agent_url was still
# the literal string '${LANGFLOW_URL:-http://...}'. We implement the
# `:-default` form directly instead of relying on os.path.expandvars.
import re

_ENV_VAR_PATTERN = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)(:-([^}]*))?\}")

def _expand_env_string(value):
    """Expand ${VAR} and ${VAR:-default} placeholders in a string."""
    def _replace(match):
        var_name = match.group(1)
        has_default = match.group(2) is not None
        default = match.group(3) if has_default else None
        env_value = os.environ.get(var_name)
        if env_value:
            return env_value
        if has_default:
            return default or ''
        return match.group(0)  # leave unexpanded if no default and unset
    return _ENV_VAR_PATTERN.sub(_replace, value)

def _expand_vars_recursive(obj):
    """Recursively expand ${VAR}/${VAR:-default} in all string values."""
    if isinstance(obj, str):
        expanded = _expand_env_string(obj)
        if '${' in expanded:
            logger.warning('Unexpanded env var: %r -- set the variable', expanded)
        return expanded
    elif isinstance(obj, dict):
        return {k: _expand_vars_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_expand_vars_recursive(item) for item in obj]
    return obj

def _load_config(path):
    """Load YAML config with ${VAR}/${VAR:-default} interpolation."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return _expand_vars_recursive(raw) if raw else {}

# Load flow config
config = _load_config('../../sdg/config/flow-training.yaml')
print('Model:', config.get('model_config', {}).get('model'))
print('API base:', config.get('model_config', {}).get('api_base'))
print('Agent framework:', config.get('agent_config', {}).get('agent_framework'))
print('Agent URL:', config.get('agent_config', {}).get('agent_url'))

In [ ]:
# ── Seed Materialization ──
# Ensure tool metadata seeds are materialized from the MCP Gateway
# before running the generation flow.

from sdg.scripts.materialize_seeds import (
    materialize_tool_seeds,
    acquire_jwt,
    mcp_initialize,
    fetch_tools,
)

import time
from pathlib import Path

seeds_output = Path('../../sdg/data/seeds/tool_metadata.jsonl')
seeds_config = '../../sdg/config/seed-scenarios.yaml'

# Check if seeds exist and are fresh (< 1 hour old)
need_materialize = True
if seeds_output.exists():
    age = time.time() - seeds_output.stat().st_mtime
    if age < 3600:
        count = sum(1 for line in seeds_output.open() if line.strip())
        print(f'Seeds file is fresh ({age:.0f}s old, {count} records) -- skipping materialization')
        need_materialize = False

if need_materialize:
    gateway_url = os.environ.get('GATEWAY_URL', 'https://mcp.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com/mcp')
    keycloak_url = os.environ.get('KEYCLOAK_URL', 'https://mcp-keycloak-otel25-gateway.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com')
    client_id = os.environ.get('CLIENT_ID', 'sdg-pipeline')
    client_secret = os.environ.get('CLIENT_SECRET', '')

    if not client_secret:
        raise ValueError('CLIENT_SECRET env var is required for seed materialization')

    jwt = acquire_jwt(keycloak_url, client_id, client_secret)
    session_id = mcp_initialize(gateway_url, jwt)
    tools = fetch_tools(gateway_url, jwt, session_id)
    count = materialize_tool_seeds(tools, seeds_config, str(seeds_output))
    print(f'Materialized {count} seed records from {len(tools)} gateway tools')

In [ ]:
# ── Load and Run Generation Flow (~100 examples across all 5 tiers) ──

from sdg_hub import FlowRegistry, Flow
import pandas as pd

FlowRegistry.discover_flows()
flow_path = FlowRegistry.get_flow_path('MCP Server Distillation')
flow = Flow.from_yaml(flow_path)

# Real Flow API (verified against installed sdg_hub==0.9.4, not the flow's
# docs, which don't cover this): model/agent config are set on the flow
# itself via set_model_config()/set_agent_config(), not passed to generate().
# generate() takes a pandas DataFrame or HF Dataset (not a list of dicts)
# and its only config kwarg is runtime_params.
flow.set_model_config(**config.get('model_config', {}))
flow.set_agent_config(**config.get('agent_config', {}))

# Dry-run runtime params: sample multiple generated variations per seed.
# 21 seeds (7 single_tool + 4 parallel + 4 sequential + 3 multi_turn + 3 closed_loop)
# x ~5 samples/seed ~= 100-105 raw examples -- capped to 100 below.
DRY_RUN_TARGET = 100
SAMPLES_PER_SEED = 5
dry_run_params = {
    'multiply_tool_rows': {'num_samples': SAMPLES_PER_SEED},
    'sample_tools': {'num_samples': 1},
}

# Load seeds from ALL curriculum tiers (not just single_tool) -- this is what
# lets the dry-run exercise the multi-turn serialization path too (D-20).
seeds = []
with open(seeds_output) as f:
    for line in f:
        seeds.append(json.loads(line.strip()))
print(f'Loaded {len(seeds)} seeds across all tiers: '
      f"{sorted(set(s['tier'] for s in seeds))}")

# sdg_hub's Flow.generate() requires a pandas DataFrame or HF Dataset, not a
# list of dicts.
seed_df = pd.DataFrame(seeds)

# Run generation
output = flow.generate(
    dataset=seed_df,
    runtime_params=dry_run_params,
)

# Convert to list of dicts
if hasattr(output, 'iterrows'):
    results = [row.to_dict() for _, row in output.iterrows()]
else:
    results = list(output)

# Cap at target -- sampling 5x per seed across 21 seeds may slightly over- or
# under-shoot 100 depending on how sdg_hub's sample_tools/filter blocks behave.
results = results[:DRY_RUN_TARGET]
print(f'Generated {len(results)} examples (target: {DRY_RUN_TARGET})')
if results:
    print(f'Columns in generated output: {sorted(results[0].keys())}')

# Save raw output
dry_run_dir = Path('../../sdg/data/raw/dry-run/')
dry_run_dir.mkdir(parents=True, exist_ok=True)
with open(dry_run_dir / 'all_tiers_raw.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r, default=str) + '\n')
print(f'Raw output saved to {dry_run_dir / "all_tiers_raw.jsonl"}')

In [ ]:
# ── Column Mapping ──
# Bridge sdg_hub output columns to serializer input keys
# Same COLUMN_MAP as run-evaluation.py, plus tier/turns passthrough for
# multi-turn routing (see _to_multi_turn_trace below).

COLUMN_MAP = {
    'question': 'question',
    'agent_text': 'answer',
    'response_text': 'answer',
    'tool_trace': 'tool_trace',
    'agent_tool_trace': 'tool_trace',
    'tool_list': 'tools',
}
REQUIRED_KEYS = {'question', 'tool_trace', 'answer', 'tools'}
PASSTHROUGH_KEYS = {'tier', 'turns'}

mapped_rows = []
for i, row in enumerate(results):
    mapped = {}
    for src, dst in COLUMN_MAP.items():
        if src in row and (dst not in mapped or mapped[dst] is None):
            mapped[dst] = row[src]
    for key in REQUIRED_KEYS:
        if key in row and key not in mapped:
            mapped[key] = row[key]
    for key in PASSTHROUGH_KEYS:
        if key in row:
            mapped[key] = row[key]

    # Parse JSON strings
    for key in ('tool_trace', 'tools'):
        if isinstance(mapped.get(key), str):
            try:
                mapped[key] = json.loads(mapped[key])
            except (json.JSONDecodeError, ValueError):
                pass

    missing = REQUIRED_KEYS - set(mapped.keys())
    if missing:
        raise ValueError(
            f'Row {i}: missing keys {missing} after mapping. '
            f'Available: {sorted(row.keys())}'
        )
    mapped_rows.append(mapped)

print(f'Column mapping complete: {len(mapped_rows)} rows mapped')
print(f'Tier distribution: ' + str({t: sum(1 for r in mapped_rows if r.get("tier") == t) for t in sorted(set(r.get("tier") for r in mapped_rows))}))

In [ ]:
# ── OSFT Serialization (both single-turn and multi-turn, D-20) ──
# Import from the proper package path (underscore filename).
# _to_multi_turn_trace mirrors run-evaluation.py's fix: multi_turn/closed_loop
# tier rows with a native sdg_hub 'turns' list use it directly; everything
# else (including multi_turn rows sdg_hub didn't populate) is wrapped as a
# trivial 1-turn record. Without this, serialize_multi_turn() on a flat row
# produces a degenerate system-message-only record (no turns key present).

from sdg.scripts.serialize_osft import serialize_single_turn, serialize_multi_turn

MULTI_TURN_TIERS = {'multi_turn', 'closed_loop'}

def _to_multi_turn_trace(row):
    if row.get('tier') in MULTI_TURN_TIERS and row.get('turns'):
        return {'turns': row['turns'], 'tools': row.get('tools', [])}
    return {
        'turns': [{
            'question': row['question'],
            'tool_trace': row['tool_trace'],
            'answer': row['answer'],
        }],
        'tools': row.get('tools', []),
    }

serialized_single = []
serialized_multi = []
for i, row in enumerate(mapped_rows):
    try:
        serialized_single.append(serialize_single_turn(row))
    except (ValueError, KeyError) as exc:
        print(f'Row {i} single-turn serialization failed: {exc}')
    try:
        serialized_multi.append(serialize_multi_turn(_to_multi_turn_trace(row)))
    except (ValueError, KeyError) as exc:
        print(f'Row {i} multi-turn serialization failed: {exc}')

print(f'Serialized {len(serialized_single)}/{len(mapped_rows)} single-turn examples')
print(f'Serialized {len(serialized_multi)}/{len(mapped_rows)} multi-turn examples')

# Validate both; downstream cells (OSFT Validation / Sample Output) operate
# on `serialized_examples` -- default to single-turn for that inspection,
# multi-turn is validated and saved separately right here.
serialized_examples = serialized_single

In [ ]:
# ── OSFT Validation (both formats, D-20 + amended D-04) ──
# Per amended D-04, OSFT validation runs automatically, same as in run-evaluation.py.
# This is NOT a separate manual step -- it validates every serialized row,
# for both the single-turn and multi-turn delivery files.

from sdg.lib.osft_validation import validate_osft_line

def _validate_set(examples, label):
    validated, failed = [], []
    for i, example in enumerate(examples):
        errors = validate_osft_line(example)
        if errors:
            failed.append({'row': i, 'errors': errors})
            print(f'[{label}] Row {i} FAILED: {errors}')
        else:
            validated.append(example)
    print(f'[{label}] Passed: {len(validated)}/{len(examples)}, Failed: {len(failed)}/{len(examples)}')
    if failed:
        print(f'[{label}] Failed rows (first 5):')
        for f in failed[:5]:
            print(f'  Row {f["row"]}: {f["errors"]}')
    return validated

validated = _validate_set(serialized_single, 'single-turn')
validated_multi = _validate_set(serialized_multi, 'multi-turn')

# Save validated output for Farbod's team -- both formats (D-20)
dry_run_output = dry_run_dir / 'dry-run.jsonl'
with open(dry_run_output, 'w') as f:
    for example in validated:
        f.write(json.dumps(example) + '\n')
print(f'\nSingle-turn validated output saved to {dry_run_output}')

dry_run_multi_output = dry_run_dir / 'dry-run-multi-turn.jsonl'
with open(dry_run_multi_output, 'w') as f:
    for example in validated_multi:
        f.write(json.dumps(example) + '\n')
print(f'Multi-turn validated output saved to {dry_run_multi_output}')

In [ ]:
# ── Sample Output ──
# Display 3 example OSFT JSONL records for visual inspection

from IPython.display import JSON, display

for i, example in enumerate(validated[:3]):
    print(f'\n--- Example {i+1} ---')
    print(f'Messages count: {len(example["messages"])}')
    for msg in example['messages']:
        role = msg['role']
        if role == 'system':
            print(f'  [{role}] {msg["content"][:80]}...')
        elif role == 'user':
            print(f'  [{role}] {msg["content"][:120]}')
        elif role == 'assistant' and msg.get('tool_calls'):
            for tc in msg['tool_calls']:
                print(f'  [{role}] tool_call: {tc["function"]["name"]}({tc["function"]["arguments"][:60]})')
        elif role == 'tool':
            print(f'  [{role}] call_id={msg["tool_call_id"][:20]}... content={msg["content"][:60]}...')
        elif role == 'assistant':
            print(f'  [{role}] {msg["content"][:120]}')
    print()
    # Full JSON for first example
    if i == 0:
        print('Full JSON:')
        print(json.dumps(example, indent=2)[:2000])

## Next Step: Send dry-run output to Farbod's team for OSFT validation

Two files are produced (D-20 -- both formats, Farbod's team decides what to train on):
- `sdg/data/raw/dry-run/dry-run.jsonl` -- single-turn format
- `sdg/data/raw/dry-run/dry-run-multi-turn.jsonl` -- multi-turn format

Farbod's team should:
1. Run `osft(data_path="dry-run.jsonl")` with Training Hub (and the multi-turn variant too, if relevant to their training plan)
2. Confirm the data tokenizes correctly
3. Confirm loss values are normal

**D-19 Gate:** Do NOT proceed to the full production run (03-production-run.ipynb)
until Farbod's team confirms the format works.

If issues are found:
- If `tool_calls` not recognized: check if `mini_trainer` needs a patch or data needs pre-tokenization
- If loss values abnormal: check token masking in the chat template